# Laughter Detection Config Comparison

Run one MP3 through the same laughter detector with multiple threshold/min_length settings. The model is evaluated once to compute probabilities, then each config reuses those probabilities and writes its own review folder in Google Drive.


In [ ]:
#@title Clone laughter-detection repository

import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/laughter-detection').resolve()

if not REPO_DIR.exists():
    subprocess.check_call([
        'git',
        'clone',
        'https://github.com/jorrytdejong/laughter-detection.git',
        str(REPO_DIR),
    ])
else:
    print(f'Using existing repo: {REPO_DIR}')

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'utils'))

print(f'Repo dir: {REPO_DIR}')


In [ ]:
#@title Install / Setup

import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages])


os.environ.setdefault('PIP_DISABLE_PIP_VERSION_CHECK', '1')
os.environ.setdefault('PIP_NO_INPUT', '1')

protobuf_pin = 'protobuf==3.20.3'

if sys.version_info >= (3, 11):
    core_pkgs = [
        'librosa>=0.10.1',
        'numpy>=1.24',
        'tgt==1.4.4',
        'pyloudnorm==0.1.0',
        'praatio==3.8.0',
        'tensorboardX==1.9',
        protobuf_pin,
    ]
else:
    core_pkgs = [
        'librosa==0.8.1',
        'numpy<1.24',
        'tgt==1.4.4',
        'pyloudnorm==0.1.0',
        'praatio==3.8.0',
        'tensorboardX==1.9',
        protobuf_pin,
    ]

pip_install(core_pkgs)

try:
    import torch
except Exception:
    pip_install(['torch'])

import numpy as np

if not hasattr(np, 'complex'):
    np.complex = complex  # type: ignore[attr-defined]

if not hasattr(np, 'float'):
    np.float = float  # type: ignore[attr-defined]

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_OUTPUT_ROOT = (
    Path('/content/drive/MyDrive/FreekdeJonge/laughter_detection_comparisons').resolve()
    if IN_COLAB
    else (Path.cwd() / 'FreekdeJonge' / 'laughter_detection_comparisons').resolve()
)

print('Setup complete.')
print(f'Comparison output root: {DRIVE_OUTPUT_ROOT}')


## GPU

In Colab, enable a GPU to speed up the model pass: Runtime > Change runtime type > Hardware accelerator > GPU.


In [ ]:
#@title Setup and load model

import os
import sys
from functools import partial
from pathlib import Path

import numpy as np
import torch
from torch.serialization import add_safe_globals

REPO_DIR = Path('/content/laughter-detection').resolve()
if not REPO_DIR.exists():
    raise FileNotFoundError('Repository not found. Run the clone cell first.')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'utils'))

os.environ.setdefault('PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION', 'python')

if not hasattr(np, 'complex'):
    np.complex = complex  # type: ignore[attr-defined]

if not hasattr(np, 'float'):
    np.float = float  # type: ignore[attr-defined]

add_safe_globals([np.core.multiarray.scalar])

import audio_utils
import configs
import data_loaders
import laugh_segmenter
import models
import torch_utils

SAMPLE_RATE = 8000
MODEL_CONFIG_NAME = 'resnet_with_augmentation'
model_path = REPO_DIR / 'checkpoints' / 'in_use' / MODEL_CONFIG_NAME
model_config = configs.CONFIG_MAP[MODEL_CONFIG_NAME]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = model_config['model'](
    dropout_rate=0.0,
    linear_layer_size=model_config['linear_layer_size'],
    filter_sizes=model_config['filter_sizes'],
)
model.set_device(device)

feature_fn = model_config['feature_fn']

checkpoint_path = model_path / 'best.pth.tar'
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Model checkpoint not found: {checkpoint_path}')

torch_utils.load_checkpoint(str(checkpoint_path), model)
model.eval()

print('Model loaded successfully.')


In [ ]:
#@title Comparison config

from datetime import datetime
from pathlib import Path

AUDIO_FILE = '/content/your_audio.mp3'  #@param {type:"string"}
AUDIO_PATH = Path(AUDIO_FILE).expanduser().resolve()

OUTPUT_ROOT = DRIVE_OUTPUT_ROOT
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

THRESHOLDS = [0.3, 0.4, 0.5]
MIN_LENGTHS = [0.1, 0.2, 0.3]

SAVE_AUDIO_FILES = True
CLEAR_EXISTING_CONFIG_OUTPUTS = True

if not AUDIO_PATH.exists():
    raise FileNotFoundError(f'Audio file not found: {AUDIO_PATH}')

RUN_ROOT = (OUTPUT_ROOT / AUDIO_PATH.stem / RUN_ID).resolve()
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Audio path: {AUDIO_PATH}')
print(f'Run root: {RUN_ROOT}')
print(f'Default configs: {len(THRESHOLDS) * len(MIN_LENGTHS)}')


In [ ]:
#@title Reusable comparison functions

import csv
import json
import shutil
from pathlib import Path

import librosa
import numpy as np
import scipy.io.wavfile
from tqdm import tqdm


def format_float_for_name(value):
    return f'{float(value):.3f}'.rstrip('0').rstrip('.')


def build_config_grid(thresholds, min_lengths):
    configs_to_run = []
    for threshold in thresholds:
        for min_length in min_lengths:
            threshold_label = format_float_for_name(threshold)
            min_length_label = format_float_for_name(min_length)
            configs_to_run.append(
                {
                    'name': f'threshold_{threshold_label}_minlen_{min_length_label}',
                    'threshold': float(threshold),
                    'min_length': float(min_length),
                }
            )
    return configs_to_run


def compute_laughter_probabilities(audio_path):
    audio_path = Path(audio_path).resolve()
    audio_path_str = str(audio_path)

    inference_dataset = data_loaders.SwitchBoardLaughterInferenceDataset(
        audio_path=audio_path_str,
        feature_fn=feature_fn,
        sr=SAMPLE_RATE,
    )
    collate_fn = partial(
        audio_utils.pad_sequences_with_labels,
        expand_channel_dim=model_config['expand_channel_dim'],
    )
    inference_generator = torch.utils.data.DataLoader(
        inference_dataset,
        num_workers=4,
        batch_size=8,
        shuffle=False,
        collate_fn=collate_fn,
    )

    probs = []
    with torch.no_grad():
        for model_inputs, _ in tqdm(inference_generator, desc='Model pass'):
            x = torch.from_numpy(model_inputs).float().to(device)
            preds = model(x).cpu().detach().numpy().squeeze()
            if len(preds.shape) == 0:
                probs.append(float(preds))
            else:
                probs.extend(list(preds))

    raw_probs = np.array(probs)
    file_length = audio_utils.get_audio_length(audio_path_str)
    fps = len(raw_probs) / float(file_length)
    smoothed_probs = laugh_segmenter.lowpass(raw_probs)

    return {
        'raw_probs': raw_probs,
        'smoothed_probs': smoothed_probs,
        'fps': fps,
        'file_length': float(file_length),
    }


def prepare_config_dir(config_dir):
    config_dir = Path(config_dir)
    if config_dir.exists() and CLEAR_EXISTING_CONFIG_OUTPUTS:
        shutil.rmtree(config_dir)
    config_dir.mkdir(parents=True, exist_ok=True)
    return config_dir


def save_laughter_wavs(instances, audio_samples, audio_sr, config_dir):
    wav_paths = []
    maxv = np.iinfo(np.int16).max

    if not SAVE_AUDIO_FILES:
        return wav_paths

    for index, instance in enumerate(instances):
        laughs = laugh_segmenter.cut_laughter_segments([instance], audio_samples, audio_sr)
        laughs = np.clip(laughs, -1.0, 1.0)
        wav_path = config_dir / f'laugh_{index:03d}.wav'
        scipy.io.wavfile.write(str(wav_path), audio_sr, (laughs * maxv).astype(np.int16))
        wav_paths.append(wav_path)

    return wav_paths


def build_segments(instances, wav_paths):
    segments = []
    for index, (start, end) in enumerate(instances):
        filename = wav_paths[index].name if index < len(wav_paths) else f'laugh_{index:03d}.wav'
        start = float(start)
        end = float(end)
        segments.append(
            {
                'index': index,
                'start': start,
                'end': end,
                'duration': end - start,
                'filename': filename,
            }
        )
    return segments


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


def write_manual_evaluation_csv(path, segments):
    headers = [
        'index',
        'start',
        'end',
        'duration',
        'filename',
        'is_laughter_manual',
        'confidence_manual',
        'notes',
    ]
    with Path(path).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        for segment in segments:
            writer.writerow(
                {
                    'index': segment['index'],
                    'start': segment['start'],
                    'end': segment['end'],
                    'duration': segment['duration'],
                    'filename': segment['filename'],
                    'is_laughter_manual': '',
                    'confidence_manual': '',
                    'notes': '',
                }
            )


def run_single_config(config_item, probability_data, audio_samples, audio_sr):
    config_dir = prepare_config_dir(RUN_ROOT / config_item['name'])
    instances = laugh_segmenter.get_laughter_instances(
        probability_data['smoothed_probs'],
        threshold=config_item['threshold'],
        min_length=config_item['min_length'],
        fps=probability_data['fps'],
    )
    wav_paths = save_laughter_wavs(instances, audio_samples, audio_sr, config_dir)
    segments = build_segments(instances, wav_paths)
    total_laughter_seconds = sum(segment['duration'] for segment in segments)

    run_config = {
        **config_item,
        'source_audio': AUDIO_PATH.name,
        'audio_path': str(AUDIO_PATH),
        'run_id': RUN_ID,
        'model_config_name': MODEL_CONFIG_NAME,
        'sample_rate': SAMPLE_RATE,
        'probability_fps': probability_data['fps'],
        'file_length_seconds': probability_data['file_length'],
        'save_audio_files': SAVE_AUDIO_FILES,
    }

    timestamps_payload = {
        'source_audio': AUDIO_PATH.name,
        'config': config_item,
        'segments': segments,
        'total_count': len(segments),
        'total_laughter_seconds': total_laughter_seconds,
    }

    timestamps_path = config_dir / f'{AUDIO_PATH.stem}_laughter_timestamps.json'
    run_config_path = config_dir / 'run_config.json'
    evaluation_path = config_dir / 'manual_evaluation.csv'

    write_json(timestamps_path, timestamps_payload)
    write_json(run_config_path, run_config)
    write_manual_evaluation_csv(evaluation_path, segments)

    return {
        'config_name': config_item['name'],
        'threshold': config_item['threshold'],
        'min_length': config_item['min_length'],
        'laugh_count': len(segments),
        'total_laughter_seconds': total_laughter_seconds,
        'output_folder': str(config_dir),
        'timestamps_json': str(timestamps_path),
        'run_config_json': str(run_config_path),
        'manual_evaluation_csv': str(evaluation_path),
    }


def write_comparison_summary(summary_rows):
    summary_path = RUN_ROOT / 'comparison_summary.csv'
    headers = [
        'config_name',
        'threshold',
        'min_length',
        'laugh_count',
        'total_laughter_seconds',
        'output_folder',
    ]
    with summary_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        for row in summary_rows:
            writer.writerow({key: row[key] for key in headers})
    return summary_path


def write_comparison_manifest(configs_to_run, summary_rows, summary_path):
    manifest_path = RUN_ROOT / 'comparison_manifest.json'
    manifest = {
        'source_audio': AUDIO_PATH.name,
        'audio_path': str(AUDIO_PATH),
        'run_id': RUN_ID,
        'run_root': str(RUN_ROOT),
        'model_config_name': MODEL_CONFIG_NAME,
        'thresholds': THRESHOLDS,
        'min_lengths': MIN_LENGTHS,
        'configs': configs_to_run,
        'summary_csv': str(summary_path),
        'results': summary_rows,
    }
    write_json(manifest_path, manifest)
    return manifest_path


In [ ]:
#@title Run comparison

configs_to_run = build_config_grid(THRESHOLDS, MIN_LENGTHS)
print(f'Running {len(configs_to_run)} config(s).')

probability_data = compute_laughter_probabilities(AUDIO_PATH)
print(
    'Computed probabilities once: '
    f"{len(probability_data['raw_probs'])} frames, "
    f"fps={probability_data['fps']:.4f}, "
    f"audio_seconds={probability_data['file_length']:.2f}"
)

audio_samples, audio_sr = librosa.load(str(AUDIO_PATH), sr=44100)
summary_rows = []

for config_item in tqdm(configs_to_run, desc='Configs'):
    summary_row = run_single_config(config_item, probability_data, audio_samples, audio_sr)
    summary_rows.append(summary_row)
    print(
        f"{summary_row['config_name']}: "
        f"{summary_row['laugh_count']} laughs, "
        f"{summary_row['total_laughter_seconds']:.2f}s"
    )

summary_path = write_comparison_summary(summary_rows)
manifest_path = write_comparison_manifest(configs_to_run, summary_rows, summary_path)

print('\nComparison complete.')
print(f'Run root: {RUN_ROOT}')
print(f'Summary CSV: {summary_path}')
print(f'Manifest JSON: {manifest_path}')


## AudioSet Taggers

Optional coarse detector cells for AST and Whisper-AT. These run over the original MP3 and write config-like review folders beside the threshold/min_length comparison outputs.


In [ ]:
#@title Install AudioSet tagger dependencies

import subprocess
import sys

try:
    pip_install
except NameError:
    def pip_install(packages):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages])

TAGGER_PACKAGES = [
    'transformers>=4.40.0',
    'soundfile',
    'whisper-at',
]

pip_install(TAGGER_PACKAGES)
print('AudioSet tagger dependencies installed.')


In [ ]:
#@title AudioSet tagger config

from pathlib import Path

AST_MODEL_ID = 'MIT/ast-finetuned-audioset-10-10-0.4593'
AST_WINDOW_SECONDS = 5.0  #@param {type:"number"}
AST_HOP_SECONDS = 2.5  #@param {type:"number"}
AST_LAUGHTER_THRESHOLD = 0.20  #@param {type:"number"}
AST_BATCH_SIZE = 8  #@param {type:"integer"}

WHISPER_AT_MODEL_NAME = 'base'  #@param {type:"string"}
WHISPER_AT_TIME_RES = 10.0  #@param {type:"number"}
WHISPER_AT_LAUGHTER_THRESHOLD = -1.0  #@param {type:"number"}

TAGGER_MERGE_GAP_SECONDS = 1.0  #@param {type:"number"}

AUDIOSET_LAUGHTER_LABELS = [
    'Laughter',
    'Baby laughter',
    'Giggle',
    'Snicker',
    'Belly laugh',
    'Chuckle, chortle',
]
AUDIOSET_LAUGHTER_INDICES = list(range(16, 22))

if not AUDIO_PATH.exists():
    raise FileNotFoundError(f'Audio file not found: {AUDIO_PATH}')

RUN_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Audio path: {AUDIO_PATH}')
print(f'Run root: {RUN_ROOT}')
print(f'AST: window={AST_WINDOW_SECONDS}s hop={AST_HOP_SECONDS}s threshold={AST_LAUGHTER_THRESHOLD}')
print(f'Whisper-AT: model={WHISPER_AT_MODEL_NAME} at_time_res={WHISPER_AT_TIME_RES}s threshold={WHISPER_AT_LAUGHTER_THRESHOLD}')


In [ ]:
#@title Shared AudioSet tagger helpers

import csv
import json
from pathlib import Path

import librosa
import numpy as np
import scipy.io.wavfile
import torch
from tqdm import tqdm


def chunked(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def score_column_name(label):
    return 'score_' + ''.join(ch.lower() if ch.isalnum() else '_' for ch in label).strip('_')


def get_or_load_full_res_audio():
    global audio_samples, audio_sr
    if 'audio_samples' not in globals() or 'audio_sr' not in globals():
        audio_samples, audio_sr = librosa.load(str(AUDIO_PATH), sr=44100)
    return audio_samples, audio_sr


def make_sliding_windows(audio_array, sampling_rate, window_seconds, hop_seconds):
    if window_seconds <= 0 or hop_seconds <= 0:
        raise ValueError('Window and hop seconds must be positive.')

    audio_length = len(audio_array)
    audio_seconds = audio_length / float(sampling_rate)
    window_size = max(1, int(round(window_seconds * sampling_rate)))
    hop_size = max(1, int(round(hop_seconds * sampling_rate)))

    if audio_length == 0:
        return []

    max_start = max(0, audio_length - window_size)
    starts = list(range(0, max_start + 1, hop_size)) or [0]
    if starts[-1] != max_start:
        starts.append(max_start)

    windows = []
    for window_index, start_sample in enumerate(sorted(set(starts))):
        end_sample = min(audio_length, start_sample + window_size)
        start_seconds = start_sample / float(sampling_rate)
        end_seconds = min(audio_seconds, end_sample / float(sampling_rate))
        windows.append(
            {
                'window_index': window_index,
                'start': start_seconds,
                'end': end_seconds,
                'audio': audio_array[start_sample:end_sample],
            }
        )
    return windows


def get_label_indices_from_id2label(id2label, labels, fallback_indices=None):
    normalized = {str(label).strip().lower(): int(index) for index, label in id2label.items()}
    label_to_index = {}
    missing = []
    for label in labels:
        key = label.strip().lower()
        if key in normalized:
            label_to_index[label] = normalized[key]
        else:
            missing.append(label)

    if missing and fallback_indices is not None:
        for label, fallback_index in zip(labels, fallback_indices):
            label_to_index.setdefault(label, int(fallback_index))
        missing = [label for label in labels if label not in label_to_index]

    if missing:
        raise ValueError(f'Could not find AudioSet label(s): {missing}')

    return label_to_index


def merge_tagger_windows(window_rows, threshold, merge_gap_seconds):
    positive_rows = [row for row in window_rows if row['score'] >= threshold]
    positive_rows = sorted(positive_rows, key=lambda row: (row['start'], row['end']))
    merged = []

    for row in positive_rows:
        if not merged or row['start'] > merged[-1]['end'] + merge_gap_seconds:
            merged.append(
                {
                    'start': float(row['start']),
                    'end': float(row['end']),
                    'score': float(row['score']),
                    'best_laughter_label': row['best_laughter_label'],
                    'source_windows': [int(row['window_index'])],
                }
            )
            continue

        current = merged[-1]
        current['end'] = max(float(current['end']), float(row['end']))
        current['source_windows'].append(int(row['window_index']))
        if row['score'] > current['score']:
            current['score'] = float(row['score'])
            current['best_laughter_label'] = row['best_laughter_label']

    for index, segment in enumerate(merged):
        segment['index'] = index
        segment['duration'] = float(segment['end'] - segment['start'])
        segment['filename'] = f'laugh_{index:03d}.wav'

    return merged


def write_tagger_window_scores_csv(path, window_rows):
    score_columns = [score_column_name(label) for label in AUDIOSET_LAUGHTER_LABELS]
    headers = ['window_index', 'start', 'end', 'score', 'best_laughter_label', *score_columns]
    with Path(path).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        for row in window_rows:
            csv_row = {
                'window_index': row['window_index'],
                'start': row['start'],
                'end': row['end'],
                'score': row['score'],
                'best_laughter_label': row['best_laughter_label'],
            }
            for label in AUDIOSET_LAUGHTER_LABELS:
                csv_row[score_column_name(label)] = row['label_scores'].get(label, '')
            writer.writerow(csv_row)


def write_tagger_manual_evaluation_csv(path, segments, detector_name):
    headers = [
        'index',
        'start',
        'end',
        'duration',
        'filename',
        'is_laughter_manual',
        'confidence_manual',
        'notes',
        'detector',
        'score',
        'best_laughter_label',
    ]
    with Path(path).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        for segment in segments:
            writer.writerow(
                {
                    'index': segment['index'],
                    'start': segment['start'],
                    'end': segment['end'],
                    'duration': segment['duration'],
                    'filename': segment['filename'],
                    'is_laughter_manual': '',
                    'confidence_manual': '',
                    'notes': '',
                    'detector': detector_name,
                    'score': segment['score'],
                    'best_laughter_label': segment['best_laughter_label'],
                }
            )


def save_tagger_wavs(segments, config_dir):
    full_res_audio, full_res_sr = get_or_load_full_res_audio()
    maxv = np.iinfo(np.int16).max
    if not SAVE_AUDIO_FILES:
        return segments

    for segment in segments:
        laugh_audio = laugh_segmenter.cut_laughter_segments(
            [(segment['start'], segment['end'])],
            full_res_audio,
            full_res_sr,
        )
        laugh_audio = np.clip(laugh_audio, -1.0, 1.0)
        wav_path = Path(config_dir) / segment['filename']
        scipy.io.wavfile.write(str(wav_path), full_res_sr, (laugh_audio * maxv).astype(np.int16))
    return segments


def write_tagger_outputs(detector_name, config_name, config_payload, window_rows, threshold, merge_gap_seconds):
    config_dir = prepare_config_dir(RUN_ROOT / config_name)
    segments = merge_tagger_windows(window_rows, threshold, merge_gap_seconds)
    segments = save_tagger_wavs(segments, config_dir)
    total_laughter_seconds = sum(segment['duration'] for segment in segments)

    timestamps_path = config_dir / f'{AUDIO_PATH.stem}_laughter_timestamps.json'
    run_config_path = config_dir / 'run_config.json'
    evaluation_path = config_dir / 'manual_evaluation.csv'
    window_scores_path = config_dir / 'window_scores.csv'

    run_config = {
        **config_payload,
        'detector': detector_name,
        'config_name': config_name,
        'source_audio': AUDIO_PATH.name,
        'audio_path': str(AUDIO_PATH),
        'run_id': RUN_ID,
        'laughter_labels': AUDIOSET_LAUGHTER_LABELS,
        'laughter_indices': AUDIOSET_LAUGHTER_INDICES,
        'merge_gap_seconds': merge_gap_seconds,
        'save_audio_files': SAVE_AUDIO_FILES,
    }
    timestamps_payload = {
        'source_audio': AUDIO_PATH.name,
        'detector': detector_name,
        'config': run_config,
        'segments': segments,
        'total_count': len(segments),
        'total_laughter_seconds': total_laughter_seconds,
    }

    write_json(timestamps_path, timestamps_payload)
    write_json(run_config_path, run_config)
    write_tagger_manual_evaluation_csv(evaluation_path, segments, detector_name)
    write_tagger_window_scores_csv(window_scores_path, window_rows)

    return {
        'detector': detector_name,
        'config_name': config_name,
        'threshold': threshold,
        'window_seconds': config_payload.get('window_seconds', ''),
        'hop_seconds': config_payload.get('hop_seconds', ''),
        'at_time_res': config_payload.get('at_time_res', ''),
        'laugh_count': len(segments),
        'total_laughter_seconds': total_laughter_seconds,
        'output_folder': str(config_dir),
        'timestamps_json': str(timestamps_path),
        'run_config_json': str(run_config_path),
        'manual_evaluation_csv': str(evaluation_path),
        'window_scores_csv': str(window_scores_path),
    }


def write_audioset_tagger_summary(summary_rows):
    summary_path = RUN_ROOT / 'audioset_tagger_summary.csv'
    headers = [
        'detector',
        'config_name',
        'threshold',
        'window_seconds',
        'hop_seconds',
        'at_time_res',
        'laugh_count',
        'total_laughter_seconds',
        'output_folder',
    ]
    with summary_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        for row in summary_rows:
            writer.writerow({key: row.get(key, '') for key in headers})
    return summary_path


def write_audioset_tagger_manifest(summary_rows, summary_path):
    manifest_path = RUN_ROOT / 'audioset_tagger_manifest.json'
    manifest = {
        'source_audio': AUDIO_PATH.name,
        'audio_path': str(AUDIO_PATH),
        'run_id': RUN_ID,
        'run_root': str(RUN_ROOT),
        'laughter_labels': AUDIOSET_LAUGHTER_LABELS,
        'laughter_indices': AUDIOSET_LAUGHTER_INDICES,
        'merge_gap_seconds': TAGGER_MERGE_GAP_SECONDS,
        'summary_csv': str(summary_path),
        'results': summary_rows,
    }
    write_json(manifest_path, manifest)
    return manifest_path


def record_tagger_result(summary_row):
    global audioset_tagger_summary_rows
    if 'audioset_tagger_summary_rows' not in globals():
        audioset_tagger_summary_rows = []

    audioset_tagger_summary_rows = [
        row for row in audioset_tagger_summary_rows
        if row.get('config_name') != summary_row.get('config_name')
    ]
    audioset_tagger_summary_rows.append(summary_row)

    summary_path = write_audioset_tagger_summary(audioset_tagger_summary_rows)
    manifest_path = write_audioset_tagger_manifest(audioset_tagger_summary_rows, summary_path)
    return summary_path, manifest_path


In [ ]:
#@title Run AST AudioSet tagger

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

ast_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ast_feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)
ast_model = AutoModelForAudioClassification.from_pretrained(AST_MODEL_ID).to(ast_device)
ast_model.eval()

ast_sampling_rate = getattr(ast_feature_extractor, 'sampling_rate', 16000)
ast_label_to_index = get_label_indices_from_id2label(
    ast_model.config.id2label,
    AUDIOSET_LAUGHTER_LABELS,
    fallback_indices=AUDIOSET_LAUGHTER_INDICES,
)

ast_audio, _ = librosa.load(str(AUDIO_PATH), sr=ast_sampling_rate, mono=True)
ast_windows = make_sliding_windows(ast_audio, ast_sampling_rate, AST_WINDOW_SECONDS, AST_HOP_SECONDS)
ast_window_rows = []

for window_batch in tqdm(list(chunked(ast_windows, AST_BATCH_SIZE)), desc='AST windows'):
    batch_audio = [window['audio'] for window in window_batch]
    inputs = ast_feature_extractor(
        batch_audio,
        sampling_rate=ast_sampling_rate,
        return_tensors='pt',
        padding=True,
    )
    inputs = {key: value.to(ast_device) for key, value in inputs.items()}

    with torch.no_grad():
        logits = ast_model(**inputs).logits
        probabilities = torch.sigmoid(logits).cpu().numpy()

    for window, probs in zip(window_batch, probabilities):
        label_scores = {
            label: float(probs[ast_label_to_index[label]])
            for label in AUDIOSET_LAUGHTER_LABELS
        }
        best_laughter_label = max(label_scores, key=label_scores.get)
        ast_window_rows.append(
            {
                'window_index': window['window_index'],
                'start': window['start'],
                'end': window['end'],
                'score': float(label_scores[best_laughter_label]),
                'best_laughter_label': best_laughter_label,
                'label_scores': label_scores,
            }
        )

ast_config_name = (
    f'audioset_ast_window_{format_float_for_name(AST_WINDOW_SECONDS)}'
    f'_hop_{format_float_for_name(AST_HOP_SECONDS)}'
    f'_threshold_{format_float_for_name(AST_LAUGHTER_THRESHOLD)}'
)
ast_summary_row = write_tagger_outputs(
    detector_name='AST AudioSet',
    config_name=ast_config_name,
    config_payload={
        'model_id': AST_MODEL_ID,
        'window_seconds': AST_WINDOW_SECONDS,
        'hop_seconds': AST_HOP_SECONDS,
        'threshold': AST_LAUGHTER_THRESHOLD,
        'batch_size': AST_BATCH_SIZE,
        'score_type': 'sigmoid_probability',
    },
    window_rows=ast_window_rows,
    threshold=AST_LAUGHTER_THRESHOLD,
    merge_gap_seconds=TAGGER_MERGE_GAP_SECONDS,
)
tagger_summary_path, tagger_manifest_path = record_tagger_result(ast_summary_row)

print('AST AudioSet tagger complete.')
print(f"Detected {ast_summary_row['laugh_count']} merged laughter segment(s).")
print(f"Output folder: {ast_summary_row['output_folder']}")
print(f'Tagger summary CSV: {tagger_summary_path}')
print(f'Tagger manifest JSON: {tagger_manifest_path}')


In [ ]:
#@title Run Whisper-AT AudioSet tagger

import whisper_at as whisper

whisper_model = whisper.load_model(WHISPER_AT_MODEL_NAME)
whisper_result = whisper_model.transcribe(str(AUDIO_PATH), at_time_res=WHISPER_AT_TIME_RES)

if 'audio_tag' not in whisper_result:
    raise KeyError('Whisper-AT result does not contain audio_tag.')

audio_tag_logits = whisper_result['audio_tag']
if hasattr(audio_tag_logits, 'detach'):
    audio_tag_logits = audio_tag_logits.detach().cpu().numpy()
else:
    audio_tag_logits = np.asarray(audio_tag_logits)

audio_length_seconds = audio_utils.get_audio_length(str(AUDIO_PATH))
whisper_window_rows = []

for window_index, logits in enumerate(audio_tag_logits):
    start_seconds = float(window_index * WHISPER_AT_TIME_RES)
    end_seconds = float(min((window_index + 1) * WHISPER_AT_TIME_RES, audio_length_seconds))
    label_scores = {
        label: float(logits[index])
        for label, index in zip(AUDIOSET_LAUGHTER_LABELS, AUDIOSET_LAUGHTER_INDICES)
    }
    best_laughter_label = max(label_scores, key=label_scores.get)
    whisper_window_rows.append(
        {
            'window_index': window_index,
            'start': start_seconds,
            'end': end_seconds,
            'score': float(label_scores[best_laughter_label]),
            'best_laughter_label': best_laughter_label,
            'label_scores': label_scores,
        }
    )

whisper_config_name = (
    f'audioset_whisper_at_{WHISPER_AT_MODEL_NAME}'
    f'_timeres_{format_float_for_name(WHISPER_AT_TIME_RES)}'
    f'_threshold_{format_float_for_name(WHISPER_AT_LAUGHTER_THRESHOLD)}'
)
whisper_summary_row = write_tagger_outputs(
    detector_name='Whisper-AT AudioSet',
    config_name=whisper_config_name,
    config_payload={
        'model_name': WHISPER_AT_MODEL_NAME,
        'at_time_res': WHISPER_AT_TIME_RES,
        'threshold': WHISPER_AT_LAUGHTER_THRESHOLD,
        'score_type': 'raw_logit',
    },
    window_rows=whisper_window_rows,
    threshold=WHISPER_AT_LAUGHTER_THRESHOLD,
    merge_gap_seconds=TAGGER_MERGE_GAP_SECONDS,
)
tagger_summary_path, tagger_manifest_path = record_tagger_result(whisper_summary_row)

print('Whisper-AT AudioSet tagger complete.')
print(f"Detected {whisper_summary_row['laugh_count']} merged laughter segment(s).")
print(f"Output folder: {whisper_summary_row['output_folder']}")
print(f'Tagger summary CSV: {tagger_summary_path}')
print(f'Tagger manifest JSON: {tagger_manifest_path}')


In [ ]:
#@title Optional: listen to one config's clips

import IPython
from IPython.display import Audio, display

CONFIG_TO_REVIEW = ''  #@param {type:"string"}

if not CONFIG_TO_REVIEW:
    available = [row['config_name'] for row in summary_rows] if 'summary_rows' in globals() else []
    if 'audioset_tagger_summary_rows' in globals():
        available += [row['config_name'] for row in audioset_tagger_summary_rows]
    print('Set CONFIG_TO_REVIEW to one of these config names:')
    for name in available:
        print(f'- {name}')
else:
    review_dir = RUN_ROOT / CONFIG_TO_REVIEW
    detected_laughs = sorted(review_dir.glob('laugh_*.wav'))
    print(f'Found {len(detected_laughs)} clip(s) in {review_dir}')
    for laugh_path in detected_laughs:
        print(laugh_path.name)
        display(Audio(str(laugh_path)))
